# Related Papers:
- https://aclanthology.org/2024.acl-short.2.pdf
- https://aclanthology.org/2024.emnlp-main.171.pdf
- https://arxiv.org/pdf/2408.09174v1
- https://aclanthology.org/2025.findings-emnlp.638.pdf
- https://aclanthology.org/2025.emnlp-main.363.pdf

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

In [2]:
import json

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_json(file_name, file):
    with open(os.path.join(REPO_PATH, "data_files", f"{file_name}.json"), "w", encoding="utf-8") as out_file:
        json.dump(file, out_file, indent=4)

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# Data Creation:

## Havest Wikipedia pages with tables:

In [91]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester()
# Run Phase 1 - Harvesting titles based on keywords
harvester.run_complete_harvest()
# Run Phase 2 - Filtering pages for tables
await harvester.filter_pages_async()

Harvesting Pages By Keywords:   0%|          | 0/233 [00:00<?, ?keyword/s]


TOTAL UNIQUE TITLES COLLECTED: 167913



Method,Keyword,Count
Prefix,List of cities,1537
Prefix,List of countries,1278
Prefix,List of towns,501
Prefix,List of villages,483
Prefix,List of districts,375
Prefix,List of municipalities,1021
Prefix,List of communes,194
Prefix,List of counties,259
Prefix,List of provinces,84
Prefix,List of territories,22


Filtering Pages with Tables:   0%|          | 0/167913 [00:00<?, ?page/s]


TOTAL PAGES WITH TABLES COLLECTED: 62306



In [93]:
save_json("table_titles", harvester.valid_titles)

In [3]:
table_titles = load_json("C:/Users/Daniel/Desktop/KnowledgeMaze/data_files/table_titles.json")

## Parse and filter Wikipedia tables:

In [271]:
from knowledge_maze.create_dataset.create_questions.table_processing import AsyncWikiExtractor, TablePostProcessor

extractor = AsyncWikiExtractor(output_file=os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
validated_tables_file_name = await extractor.run_with_progress(table_titles)
validated_tables = load_jsonl(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))

processor = TablePostProcessor(os.path.join(REPO_PATH, "data_files/final_tables.jsonl"))
cleaned_data = processor.process()

Pre-processing: cluster similar pages and discard similarities ---> 6426 pages discarded.


Extracting Tables:   0%|          | 0/55880 [00:00<?, ?Page/s]

Starting Extraction: 55880 titles | Async Limit: 20 | CPU Workers: 19
TOTAL PAGES WITH VALID TABLES: 32076 OUT OF 55880 (57.4%)


Filtering & Cleaning:   0%|          | 0/94101 [00:00<?, ?it/s]


════════════════════════════════════════════
📊 TABLE PROCESSING SUMMARY (Limit: 12)
════════════════════════════════════════════
Total tables scanned:      94101
Dropped (Bad Headers):     8435
Dropped (Cell too long):   2841
Dropped (Repeat Rows):     1783
Short tables found:        39837
Unique source pages:       16940
Final sampled (1/page):    16940
════════════════════════════════════════════

------------------------------------------------------------------------------------------------------------------------------------------------------
                                                                  ↓ EXAMPLE TABLES ↓                                                                  
------------------------------------------------------------------------------------------------------------------------------------------------------

══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════

,Rank,Player,Career,Caps,Goals
0,1,Christine Sinclair,2000–2023,331,190
1,2,Sophie Schmidt,2005–2023,226,20
2,3,Diana Matheson,2003–2021,206,19
3,4,Desiree Scott,2010–2025,188,0
4,5,Rhian Wilkinson,2003–2017,181,7
5,6,Kadeisha Buchanan,2013–present,154,6
6,7,Ashley Lawrence,2013–present,151,8
7,7,Jessie Fleming,2013–present,151,20
8,9,Brittany Baxter,2002–2014,132,5
9,9,Andrea Neil,1991–2007,132,24



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Manchester City F.C. 3–2 Queens Park Rangers F.C. > Match > Statistics
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Statistic,Manchester City,Queens Park Rangers
0,Goals scored,3,2
1,Possession,81.3%,18.7%
2,Shots on target,15,3
3,Shots off target,10,0
4,Blocked shots,19,0
5,Corner kicks,19,0
6,Fouls,4,7
7,Tackles,15,22
8,Offsides,1,1
9,Yellow cards,2,1



══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
Swinburne University of Technology > Notable people > Academics and staff
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════


,Member,Executive team,Period
0,Pascale Quester,Vice-Chancellor,2020–present
1,Chris Pilgrim,Pro Vice-Chancellor (Academic),
2,Tara Magdalinski,Pro Vice-Chancellor (Education and Quality),
3,Sarah Maddison,Pro Vice-Chancellor (Academic Innovation and C...,
4,Bronwyn Fox,Deputy Vice-Chancellor (Research and Enterprise),
5,Beth Webster,Pro Vice-Chancellor (Research Policy and Impact),
6,Qing-Lon Han,Pro Vice-Chancellor (Research Quality),
7,Alan Kin-Tak Lau,Pro Vice-Chancellor (International Research De...,
8,Blair Kuys,Dean of the School of Design and Architecture,
9,Matthew Bailes,Director of the Data Science Research Institut...,


## Create Potential Entity Groups:

In [642]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import TableGroupConstructor

group_constructor = TableGroupConstructor(random.sample(cleaned_data, 100))
await group_constructor.get_groups()

🚀 Grouping 100 tables using Entity-Focus logic...


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/100 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════════════════════════
                               📊 GROUPING REPORT                                
════════════════════════════════════════════════════════════════════════════════
  ✅ Valid Groups:   86
  ❌ Invalid Tables: 14
════════════════════════════════════════════════════════════════════════════════

✨ SAMPLE VALID NAMES:
  → Companies owned by Bonheur
  → Rectors of Iberoamerican University of Science and Technology (Chile)
  → Pumped-storage hydroelectric power stations in California

────────────────────────────────────────────────────────────────────────────────
                     🕵️ REJECTION DEBUGGER (Random Sample)                      
────────────────────────────────────────────────────────────────────────────────

[TABLE]: Demographics of Nepal > Vital statistics > Demographic and Health Survey (Ideal family size by gender and age group)
[REASON]: The table contains demographic categories (age groups) rat

In [643]:
for sample in random.sample(group_constructor.valid_tables, 25):
    print(f"Group name: {sample['Group name']}")
    print(f"Wikipedia page: {sample['Wikipedia link']}")
    print(f"Entity column: {sample['Entity column']}")
    display(pd.DataFrame(sample['Table data']))
    print()
    print()

Group name: Football clubs in the 1929–30 Football League Third Division North season
Wikipedia page: https://en.wikipedia.org/wiki/1929–30_York_City_F.C._season#Match_details
Entity column: Club


,Pos,Club,Pld,W,D,L,F,A,GA,Pts
0,4th,Chesterfield,42,22,6,14,76,56,1.36,50
1,5th,Lincoln City,42,17,14,11,83,61,1.36,48
2,6th,York City,42,15,16,11,77,64,1.2,46
3,7th,South Shields,42,18,10,14,77,74,1.04,46
4,8th,Hartlepools United,42,17,11,14,81,74,1.09,45
5,Key,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...
6,Source,,,,,,,,,




Group name: Players transferred to Birmingham City F.C. in 2013–14 season
Wikipedia page: https://en.wikipedia.org/wiki/2013–14_Birmingham_City_F.C._season#Transfers
Entity column: Player


,Date,Player,Club †,Fee
0,1 July 2013,Tom Adeyemi,( Norwich City ),Free
1,1 July 2013,Neal Eardley,( Blackpool ),Free
2,1 July 2013,Lee Novak,( Huddersfield Town ),Free
3,1 July 2013,Darren Randolph,( Motherwell ),Free
4,1 July 2013,Andrew Shinnie,( Inverness Caledonian Thistle ),Free
5,3 July 2013,Matt Green,( Mansfield Town ),Free
6,31 October 2013,Dariusz Dudka,( Levante ),Free
7,14 January 2014,Brian Howard,( CSKA Sofia ),Free
8,30 January 2014,Aaron Martin,( Southampton ),Free




Group name: Lighthouses in Sabah, Malaysia
Wikipedia page: https://en.wikipedia.org/wiki/List_of_lighthouses_in_Malaysia#Sabah
Entity column: Lighthouse


...

,Company,Sector,Ownership %
0,Fred. Olsen Cruise Lines,Cruise,100%
1,Fred. Olsen Ocean,Shipping/Offshore wind,100%
2,Fred. Olsen Renewables,Renewable energy,100%
3,Fred. Olsen Windcarrier,Shipping/Offshore wind,100%
4,Fred. Olsens gate 2,Real estate,100%
5,Koksa Eiendom,Real estate,12.6%
6,DN Media Group,Media,54%
7,Stavenes Byggeselskap,Real estate,100%




Group name: Freight cab units of EMD locomotives
Wikipedia page: https://en.wikipedia.org/wiki/List_of_EMD_locomotives#Freight_cab_units
Entity column: Model designation


,Model designation,Build year,Total produced,AAR wheel arrangement,Prime mover,Power output
0,FT,1939–1945,"555 A units , 541 B units",B-B ( B-B+B-B with B unit),EMD 16-567 EMD 16-567A,"2,700 hp (with B unit)"
1,F2,1946,"74 A units , 30 B units",B-B,EMD 16-567B,"1,350 hp (1000 kW)"
2,F3,1946–1949,"1,111 A units , 696 B units",B-B,EMD 16-567B,"1,500 hp (1,100 kW)"
3,F7,1949–1953,"2,366 A units , 1,483 B units",B-B,EMD 16-567B,"1,500 hp (1,100 kW)"
4,FP7,1949–1953,"381 A units , no B units",B-B,EMD 16-567B,"1,500 hp (1,200 kW )"
5,F9,1953–1960,"99 A units , 156 B units",B-B,EMD 16-567C,"1,750 hp (1200 kW)"
6,FP9,1954–1959,"90 A units , no B units",B-B,EMD 567C,"1,750 hp (1,300 kW )"
7,FL9,1956–1960,"60 A units , no B units",B-A1A,EMD 567C or EMD 567 D1; plus 660 V DC (3rd rail),"567C: 1,750 hp (1,300 kW ); 567D1: 1,800 hp (1..."




Group name: 1998–99 Football League First Division clubs performance
Wikipedia page: https://en.wikipedia.org/wiki/1998–99_Birmingham_City_F.C._season#Football_League_First_Division
Entity column: Club


,Pos,Club,Pld,W,D,L,F,A,GD,Pts
0,2nd,Bradford City,46,26,9,11,82,47,35,87
1,3rd,Ipswich Town,46,26,8,12,69,32,37,86
2,4th,Birmingham City,46,23,12,11,66,37,29,81
3,5th,Watford,46,21,14,11,65,56,9,77
4,6th,Bolton Wanderers,46,20,16,10,78,59,19,76
5,Key,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...,Pos = League position; Pld = Matches played; W...
6,Source,,,,,,,,,


## Create FanOut-style Questions out of the Entity Groups:

### Rule-Based Strategy Candidate Selection:

In [644]:
from knowledge_maze.create_dataset.create_questions.question_generation import StrategySelector

selector = StrategySelector(group_constructor.valid_tables)
selector.get_strategies()

Sanitizing tables:   0%|          | 0/86 [00:00<?, ?it/s]

Slicing strategy views:   0%|          | 0/86 [00:00<?, ?it/s]


✅ Strategy Selection Complete.

═══════════════════════════════════════════════════════
📊 STRATEGY SELECTION SUMMARY
-------------------------------------------------------
✅ Groups Passed:     82
❌ Groups Discarded:  4
📈 Success Rate:      95.3%
-------------------------------------------------------
Strategy Distribution (Top 4 per table):
  - UNIQUE_DIVERSITY          : 55
  - EXTREME_TEMPORAL          : 36
  - COUNT_TOTAL               : 34
  - FILTER_COUNT_TEMPORAL     : 28
  - FREQUENCY_MODE            : 21
  - AGGREGATION               : 19
  - COMPARATIVE_DIFFERENCE    : 18
  - TOP_K_RANKING             : 15
  - FILTER_COUNT_NUMERICAL    : 13
  - CATEGORICAL_FILTER_LOOKUP : 12
  - EXTREME_NUMERICAL         : 11
-------------------------------------------------------
POSSIBLE STRATEGIES  | GROUP COUNT  | PERCENTAGE
-------------------------------------------------------
#1                   | 9            |      11.0%
#2                   | 14           |      17.1%
#3         

In [597]:
selector.debug_strategy_samples("FREQUENCY_MODE")

Found 204 total tables with this strategy. Showing 5 random samples.



🔗 Source: https://en.wikipedia.org/wiki/List_of_Academy_Awards_ceremonies#Nominated_hosts


,Host,Category
0,David Niven,Best Actor
1,Michael Caine,Best Actor
2,Walter Matthau,Best Actor
3,Paul Hogan,Best Original Screenplay
4,James Franco,Best Actor
5,Seth MacFarlane,Best Original Song


🔗 Source: https://en.wikipedia.org/wiki/List_of_gig_economy_companies#List_of_gig_economy_companies


,Company,Based in
0,Fiverr,Israel
1,Airtasker,Australia
2,Amazon Mechanical Turk,United States
3,Toloka,United States
4,Figure Eight Inc.,United States
5,Freelancer.com,Australia
6,Upwork,United States
7,PeoplePerHour,United Kingdom
8,Toptal,United States
9,Hello Alfred,United States


...

,Churches,Founded
0,"St Andrew, Aikton",Medieval
1,"St Mary, Beaumont",Medieval
2,"St Michael, Burgh by Sands",Medieval
3,"St Giles, Great Orton",Medieval
4,"St Peter, Kirkbampton",Medieval
5,"St Michael, Bowness-on-Solway",Medieval
6,"St Kentigern, Caldbeck",Ancient
7,"St Mary, Sebergham",Medieval
8,"St Kentigern, Castle Sowerby",Medieval
9,"St Hilda, Westward",1560


🔗 Source: https://en.wikipedia.org/wiki/List_of_earthquakes_in_1970#Overall


,Location,MMI
0,"Peru , offshore northern Peru",X ( Extreme )
1,"China , Yunnan Province",X ( Extreme )
2,"Turkey , Kutahya Province",X ( Extreme )
3,"Iran , Golestan Province",VIII ( Severe )
4,"Soviet Union , Dagestan , Russia",VII ( Very strong )
5,"Peru , Tumbes Region",X ( Extreme )
6,"India , Gujarat",( )
7,"Australia , Madang Province , Papua and New Gu...",VIII ( Severe )
8,"Philippines , Luzon",VIII ( Severe )
9,"Peru , Huanuco Region",IX ( Violent )


🔗 Source: https://en.wikipedia.org/wiki/List_of_American_films_of_2024#Box_office


,Title,Distributor
0,Inside Out 2,Disney
1,Deadpool & Wolverine,Disney
2,Wicked,Universal
3,Moana 2,Disney
4,Despicable Me 4,Universal
5,Beetlejuice Beetlejuice,Warner Bros.
6,Dune: Part Two,Warner Bros.
7,Twisters,Universal
8,Mufasa: The Lion King,Disney
9,Sonic the Hedgehog 3,Paramount


### Model-Based Question Generation:

In [645]:
from knowledge_maze.create_dataset.create_questions.question_generation import TableSanitizer, StrategySelector, QuestionGenerator

generator = QuestionGenerator(selector.updated_data,
                              strategy_definitions=load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await generator.generate_questions()

🚀 Starting Question Generation for 82 tables...
--- Round 1: Processing 82 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/82 [00:00<?, ?it/s]

--- Round 2: Processing 3 tables ---


Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/3 [00:00<?, ?it/s]

--- Round 3: Processing 2 tables ---


...

-----------------------------------------------------------------

[2] Table: Matches in the 1988–89 Associate Members' Cup Southern Section First Round
    Original Data: https://en.wikipedia.org/wiki/1988–89_Associate_Members'_Cup#First_round
    Strategy Attempted: EXTREME_TEMPORAL
    Rejection Reason:   There is a tie for the earliest date (multiple matches on 17 January), preventing a single absolute answer.
    Table Data:


,Home team,Date
0,Brentford,17 January
1,Chesterfield,17 January
2,Colchester United,17 January
3,Northampton Town,17 January
4,Torquay United,17 January
5,Reading,18 January
6,Bristol Rovers,24 January
7,Wolverhampton Wanderers,24 January


-----------------------------------------------------------------

[3] Table: Chennai City FC Kit Manufacturers and Shirt Sponsors
    Original Data: https://en.wikipedia.org/wiki/Chennai_City_FC#Kit_manufacturers_and_shirt_sponsors
    Strategy Attempted: COUNT_TOTAL
    Rejection Reason:   The data does not provide a list of distinct entities (kit manufacturers and shirt sponsors), only periods are mentioned.
    Table Data:


,Period
0,2016—2017
1,2017—2018
2,2018—2019
3,2019—2021


-----------------------------------------------------------------


## Question Validation:

### Model-Based Question Validator (Gemini 2.5 Flash):

In [646]:
from knowledge_maze.create_dataset.create_questions.question_validator import QuestionValidator
validator = QuestionValidator(generator.valid_results, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
await validator.validate()

Usage cost: 0.0000$ (Estimated: 0.00$):   0%|          | 0/73 [00:00<?, ?it/s]

Event
Feast of the Black Nazarene
Feast of the Black Nazarene
Celebration of General Douglas MacArthur after he was relieved of duty by President Harry S. Truman
Martyrdom anniversary of Imam Hussein
Mass on the Feast of Santo Niño at the apostolic and state visit of Pope Francis in the Philippines
Feast of the Black Nazarene
Mid-Sha'ban celebration
Feast of the Black Nazarene
Santo Niño de Cebú
Mid-Sha'ban pilgrimage to the Shrine of Imam Hussein


Group,"Sold vehicles (2018), Sold vehicles (2019)"
Toyota,None
Volkswagen Group,None
General Motors (except SAIC-GM-Wuling ),None
Hyundai,None
Ford,None
Nissan,None
Honda,None
Fiat-Chrysler (now part of Stellantis ),None
Renault,None
PSA Group (now part of Stellantis ),None


Nominee(s),Year
"Tim White, Trevor White, and Will Smith",2022
Will Smith,2022
Aunjanue Ellis,2022
Zach Baylin,2022
Pamela Martin,2022
"Beyoncé and DIXSON (for "" Be Alive "")",2022
Will Smith,2023
Emancipation,2023
The cast of Emancipation,2023
Antoine Fuqua,2023


Competition,Year
World Cross Country Championships,1984
World Cross Country Championships,1985
Commonwealth Games,1986
European Championships,1986
World Championships,1987
World Cross Country Championships,1988
Olympic Games,1988


Team v t e,Pts
"Bournemouth (C, P)",90
Watford (P),89
"Norwich City (O, P)",86
Middlesbrough,85
Brentford,78


Team v t e,Pts
Charlton Athletic,40
Everton,40
Stoke City,38
Middlesbrough,37
Bolton Wanderers,37


Nominee / work,Year
"""Flori feat. Tingulli 3'nt. - Njëherë në jetë""",2007
"""Flori feat. Tingulli 3'nt. - Njëherë në jetë""",2007
"""Flori feat. Tingulli 3'nt. - Njëherë në jetë""",2007
"""Flori feat. Tingulli 3'nt. - Njëherë në jetë""",2007
"""Flori ft. Albatrit Muqiqi & Noga – Tequila Vava""",2012
"""Flori ft. Albatrit Muqiqi & Noga – Tequila Vava""",2012
"""Pa jete""",2013
"""Pa jete""",2013
"""Pa jete""",2013
"""Pa jete""",2013


Player,Status
Krešimir Ćosić,"Retired, played his entire career overseas"
Franjo Arapović,"Retired, played his entire career overseas"
Josip Sesar,"Retired, played his entire career overseas"
Stanko Barać,"Retired, played his entire career overseas"
Ante Tomić,"Currently playing for Joventut Badalona , Spain"
Goran Suton,"Retired, played his entire career overseas"
Tomislav Zubčić,"Currently playing for Napoli Basket , Italy"


### Manual Validation:

In [647]:
tagger = ManualTagger(validator.approved_data, load_json(os.path.join(REPO_PATH, "knowledge_maze/create_dataset/create_questions/question_strategies.json")))
tagger.display() # Render the output area

📦 Progress Loaded: 0 samples already tagged in 'tagger_progress.json'.


Output()

# Tests:

In [543]:
from difflib import SequenceMatcher
import pandas as pd
from IPython.display import display, HTML

def audit_categorical_system(samples, fuzzy_threshold=0.9):
    categorical_strategies = ['UNIQUE_DIVERSITY', 'FREQUENCY_MODE', 'CATEGORICAL_FILTER_LOOKUP']
    
    # Filter targets
    target_samples = [s for s in samples if s.get('Selected strategy') in categorical_strategies]
    total_samples = len(target_samples)
    
    if total_samples == 0:
        print("🔍 No samples found for the specified categorical strategies.")
        return

    problematic_sample_indices = []
    column_stats = {}
    fail_details = []

    # --- PROCESSING ---
    for idx, sample in enumerate(target_samples):
        table = sample.get('Relevant table data', [])
        if not table or not isinstance(table, list): continue
        
        strat = sample.get('Selected strategy')
        all_cols = list(table[0].keys())
        if len(all_cols) <= 1: continue
        
        # Skip the first column (Entity Column)
        data_cols = all_cols[1:] 
        sample_has_issue = False
        
        for col in data_cols:
            if col not in column_stats:
                column_stats[col] = {'systemic_fails': 0, 'any_issue_count': 0}
            
            raw_values = [str(row.get(col, "")).strip() for row in table if row.get(col)]
            unique_vals = sorted(list(set(raw_values)), key=len)
            if len(unique_vals) < 2: continue
            
            issue_vals = set()
            for i in range(len(unique_vals)):
                for j in range(i + 1, len(unique_vals)):
                    v1, v2 = unique_vals[i].lower(), unique_vals[j].lower()
                    if v1 in v2 or SequenceMatcher(None, v1, v2).ratio() >= fuzzy_threshold:
                        issue_vals.update([unique_vals[i], unique_vals[j]])

            if issue_vals:
                column_stats[col]['any_issue_count'] += 1
                sample_has_issue = True
                
                # Check for Systemic Fail
                if len(issue_vals) == len(unique_vals):
                    column_stats[col]['systemic_fails'] += 1
                    fail_details.append({
                        'sample': sample,
                        'bad_col': col,
                        'bad_values': unique_vals
                    })

        if sample_has_issue:
            problematic_sample_indices.append(idx)

    # --- 1. THE BIG PICTURE SUMMARY ---
    clean_count = total_samples - len(problematic_sample_indices)
    
    display(HTML(f"""
    <div style="border: 2px solid #2c3e50; border-radius: 10px; padding: 20px; font-family: sans-serif; background-color: #f8f9fa;">
        <h2 style="margin-top: 0; color: #2c3e50;">📊 CATEGORICAL AUDIT: TOTAL PICTURE</h2>
        <div style="display: flex; gap: 40px;">
            <div style="text-align: center;">
                <h1 style="color: #27ae60; margin: 0;">{clean_count}</h1>
                <p style="color: #666;">CLEAN SAMPLES</p>
            </div>
            <div style="text-align: center;">
                <h1 style="color: #e74c3c; margin: 0;">{len(problematic_sample_indices)}</h1>
                <p style="color: #666;">PROBLEMATIC SAMPLES</p>
            </div>
            <div style="text-align: center; border-left: 1px solid #ccc; padding-left: 40px;">
                <h1 style="color: #2980b9; margin: 0;">{(clean_count/total_samples)*100:.1f}%</h1>
                <p style="color: #666;">SUCCESS RATE</p>
            </div>
        </div>
    </div>
    """))

    # --- 2. DRILL DOWN (SYSTEMIC FAILS) ---
    if fail_details:
        display(HTML("<h3 style='margin-top: 30px; color: #c0392b;'>🚨 SYSTEMIC FAILURES DETECTED</h3>"))
        
        for fail in fail_details:
            s = fail['sample']
            df_full = pd.DataFrame(s['Relevant table data'])
            
            # Highlight the problematic column in the display
            html_output = f"""
            <div style="margin-bottom: 40px; border-left: 5px solid #e74c3c; padding-left: 15px;">
                <p style="margin: 0;"><b>Strategy:</b> <span style="color: #2980b9;">{s['Selected strategy']}</span></p>
                <p style="margin: 5px 0;"><b>Question:</b> <span style="font-size: 1.1em; color: #2c3e50;">{s['Question']}</span></p>
                <p style="margin: 0; font-size: 0.9em; color: #c0392b;"><b>Issue:</b> Every value in column <b>'{fail['bad_col']}'</b> is a near-duplicate.</p>
                <div style="margin-top: 10px;">
                    {df_full.to_html(index=False)}
                </div>
            </div>
            <hr style="border: 0; border-top: 1px solid #eee;">
            """
            display(HTML(html_output))
    else:
        print("✅ No systemic failures found in the categorical attributes.")

# Usage
# audit_categorical_system(generator.valid_results)

In [596]:
audit_categorical_system(generator.valid_results)

Officeholder,Title
Marcel Tshishiku,Extraordinary commissioner
Georges Grenfell (b. 1908),President
Paul Isombuma,President
François Aradjabu,Extraordinary commissioner-general for the former province of Haut-Congo
Alphonse Abikwa,President
Ignace Michel Alamazani,"President (from 1965, governor)"
Pierre Efomi,Governor


Title,Films
Fantastic Four 2-Movie Collection,"Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer"
The Ultimate Heroes Collection,"Daredevil , Elektra , Fantastic Four (2005), X-Men"
Marvel Heroes,"Daredevil , Elektra , Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer , X-Men , X2 , X-Men: The Last Stand"
Blu-ray 3-Pack,"Daredevil , Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer"
20th Century Fox Triple Feature,"Daredevil , Fantastic Four (2005), X-Men"
20th Century Fox Triple Feature,"Daredevil , Elektra , Fantastic Four (2005)"
20th Century Fox Double Feature,"Fantastic Four (2005), X-Men"
Fantastic Four / Fantastic Four: Rise of the Silver Surfer,"Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer"
Fantastic Four Double Feature,"Fantastic Four (2005), Fantastic Four: Rise of the Silver Surfer"


In [570]:
cou = 0
tot = 0
sam = 0
for s in [a for a in generator.valid_results if a['Selected strategy'] == "TOP_K_RANKING"]:
    if "three" in s['Question'].lower():
       tot += 1
    sam += 1
print(f"Top 3: {tot/sam}")

Top 3: 1.0


In [557]:
cou = 0
tot = 0
sam = 0
for s in [a for a in generator.valid_results if a['Selected strategy'] == "AGGREGATION"]:
    if "total" in s['Question'].lower():
       tot += 1
    if "average" in s['Question'].lower():
       cou += 1
    sam += 1
print(f"Sum: {tot/sam}, Avg: {cou/sam}")

Sum: 0.9148936170212766, Avg: 0.0851063829787234


In [595]:
import pandas as pd
import re
from IPython.display import display, HTML

def audit_numeric_in_categorical(samples):
    categorical_strategies = ['UNIQUE_DIVERSITY', 'FREQUENCY_MODE', 'CATEGORICAL_FILTER_LOOKUP']
    target_samples = [s for s in samples if s.get('Selected strategy') in categorical_strategies]
    total_samples = len(target_samples)
    
    if total_samples == 0:
        print("🔍 No samples found for categorical strategies.")
        return

    problem_details = []
    problematic_sample_indices = set()

    # Regex to find: Integers, Decimals, and Ranges (e.g., 10-20, 10 to 20, 10/20)
    # This also handles currency symbols and commas
    num_pattern = re.compile(r'^[\$€£%]*\d+([.,]\d+)?([\s\-\/]+(to[\s\-]+)?\d+([.,]\d+)?[\$€£%]*)?$')

    for idx, sample in enumerate(target_samples):
        table = sample.get('Relevant table data', [])
        if not table or not isinstance(table, list): continue
        
        all_cols = list(table[0].keys())
        if len(all_cols) <= 1: continue
        
        data_cols = all_cols[1:]
        
        for col in data_cols:
            values = [str(row.get(col, "")).strip() for row in table if row.get(col)]
            if not values: continue
            
            # Find values that match numeric or range patterns
            numeric_values = [v for v in values if num_pattern.match(v.replace(',', ''))]

            # NEW THRESHOLD: 25% (1/4 of the column)
            if len(numeric_values) >= len(values) / 4:
                problematic_sample_indices.add(idx)
                problem_details.append({
                    'sample': sample,
                    'col': col,
                    'found_nums': list(set(numeric_values))
                })
                break 

    # --- 1. THE BIG PICTURE SUMMARY ---
    clean_count = total_samples - len(problematic_sample_indices)
    
    display(HTML(f"""
    <div style="border: 2px solid #8e44ad; border-radius: 10px; padding: 20px; font-family: sans-serif; background-color: #f4f0f7;">
        <h2 style="margin-top: 0; color: #8e44ad;">🔢 NUMERIC & RANGE AUDIT (Threshold: 25%)</h2>
        <div style="display: flex; gap: 40px;">
            <div style="text-align: center;">
                <h1 style="color: #27ae60; margin: 0;">{clean_count}</h1>
                <p style="color: #666;">CATEGORICAL PURITY</p>
            </div>
            <div style="text-align: center;">
                <h1 style="color: #e74c3c; margin: 0;">{len(problematic_sample_indices)}</h1>
                <p style="color: #666;">NUMERIC/RANGE LEAKS</p>
            </div>
            <div style="text-align: center; border-left: 1px solid #ccc; padding-left: 40px;">
                <h1 style="color: #8e44ad; margin: 0;">{(clean_count/total_samples)*100:.1f}%</h1>
                <p style="color: #666;">OVERALL QUALITY</p>
            </div>
        </div>
    </div>
    """))

    # --- 2. EXAMPLES (QUESTION + FULL TABLE) ---
    if problem_details:
        display(HTML("<h3 style='margin-top: 30px; color: #e67e22;'>⚠️ LEAKAGE EXAMPLES (Including Ranges)</h3>"))
        for p in problem_details:
            s = p['sample']
            df_full = pd.DataFrame(s['Relevant table data'])
            
            html_output = f"""
            <div style="margin-bottom: 40px; border-left: 5px solid #e67e22; padding-left: 15px;">
                <p style="margin: 0;"><b>Strategy:</b> {s['Selected strategy']}</p>
                <p style="margin: 5px 0;"><b>Question:</b> <span style="font-size: 1.1em; color: #2c3e50; font-weight: bold;">{s['Question']}</span></p>
                <p style="margin: 0; color: #d35400;"><b>Issue:</b> Column <b>'{p['col']}'</b> leaked numeric values or ranges: <code style="background: #fff; padding: 2px 5px;">{p['found_nums']}</code></p>
                <div style="margin-top: 10px;">
                    {df_full.to_html(index=False)}
                </div>
            </div>
            <hr style="border: 0; border-top: 1px solid #eee;">
            """
            display(HTML(html_output))
    else:
        print("✅ No numeric or range leakage detected in categorical columns!")

In [594]:
audit_numeric_in_categorical(generator.valid_results)

College Name,NIRF Rank (2025)
Kristu Jayanti (Deemed to be University ),34
"M S Ramaiah College of Arts, Science, and Commerce",67
St Aloysius (Deemed to be University),73
St. Joseph's College of Commerce,98
Mount Carmel College,NIL
Jyoti Nivas College,NIL
NMKRV College for Women,NIL
"MES College of Arts, Commerce and Science",NIL
Maharani Lakshmi Ammanni College for Women,NIL
Seshadripuram College,NIL
